# Pipeline walkthrough — one PDF, one stage at a time

This notebook runs a **single sample PDF** through each stage and shows every
intermediate result: the raw page text, the exact prompt, the model's structured
response, the normalized rows, and the checks.

Requires `OPENAI_API_KEY` in `.env` (one API call, sub-cent). For the full batch run
and output review, see [demo.ipynb](demo.ipynb).

In [7]:
from pathlib import Path
import pandas as pd
from src.pipeline import load_settings

settings = load_settings()
SAMPLE = Path("data/NovaCloud_Q2_2025.pdf")  # any report PDF works
print("model:", settings["model"])
print("canonical metrics:", list(settings["metric_definitions"]))

model: gpt-5.4-mini
canonical metrics: ['revenue', 'arr', 'gross_margin', 'net_revenue_retention', 'logo_churn', 'headcount', 'cash', 'net_burn']


## Stage 1 — Ingest: PDF → text per page (`src/ingest.py`)

In [2]:
from src.ingest import ingest_pdf

pages = ingest_pdf(SAMPLE)
print(f"{len(pages)} page(s), {sum(len(p) for p in pages)} characters\n")
print(pages[0][:900])  # first 900 chars of page 1 — this is ALL the pipeline sees

1 page(s), 1690 characters

NovaCloud Analytics Inc.
Portfolio Company Update | B2B SaaS – Analytics & Observability
Period: Q2 2025
NovaCloud Analytics Inc. ("NovaCloud") is a B2B SaaS platform providing cloud-based analytics and observability tools to
mid-market e-commerce and digital businesses across North America and Western Europe.
Key Financial & Operating Metrics
Metric Q2 2025
Recognized Revenue (USD) $8.4M
ARR (End of Period) $34.2M
Gross Margin 78%
Net Dollar Retention 123%
Logo Churn (LTM) 5.8%
ARR per Full-Time Employee $241k
Cash Balance $19.6M
Monthly Net Burn ($0.75M)
Total Headcount 142
Recognized Revenue in Q2 2025 was $8.4M, up 6% quarter-over-quarter and 32% year-over-year. End-of-period ARR reached
$34.2M. Gross margin improved to 78% as NovaCloud continued optimising its cloud infrastructure. New logo additions totalled 14
in the quarter. Expansion revenue of $2.0M represented 24% of recognize


## Stage 2 — The prompt: rules + metric definitions + page-tagged text (`src/extract.py`)

In [3]:
from src.extract import build_prompt

system, doc_text = build_prompt(pages, settings["metric_definitions"])
print(system)  # the full instruction set, including the YAML definitions
print(doc_text) 

You are extracting metrics from a portfolio company's quarterly reporting package for an
investment firm. Extract EVERY metric you can find — from tables, from commentary prose,
and from footnotes. Rules:

- verbatim_label and value must be copied EXACTLY as printed in the document.
- verbatim_label is the label ONLY — never append the value to it ("Monthly Net Burn",
  not "Monthly Net Burn ($0.55M)").
- Copy values with signs and parentheses intact: a value printed as ($0.75M) is "($0.75M)",
  never "$0.75M".
- verbatim_label is the metric's short label (e.g. "Recognized Revenue (USD)"), NEVER a full
  sentence. For a metric that appears only in commentary prose, use the shortest exact phrase
  from the text that names the metric (e.g. "Gross Margin"), still copied verbatim.
- If the document covers MULTIPLE companies (e.g. a portfolio summary), set the company
  field on every metric to the specific company it belongs to. company_name at the document
  level is then the issuing enti

## Stage 3 — Extract: one structured-output model call

In [4]:
from src.extract import extract_document

extraction = extract_document(SAMPLE, pages, settings)
print(extraction.company_name, "|", extraction.report_period, "|", len(extraction.metrics), "metrics")

# The model's response is FORCED into this schema — it cannot return free text
raw = pd.DataFrame([m.model_dump() for m in extraction.metrics])
raw[["verbatim_label", "value", "canonical_metric", "currency", "period", "period_basis", "page", "location"]]

16:37:42 INFO    httpx2: HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


NovaCloud Analytics Inc. | Q2 2025 | 23 metrics


,verbatim_label,value,canonical_metric,currency,period,period_basis,page,location
0,Recognized Revenue (USD),$8.4M,revenue,USD,Q2 2025,quarterly,1,table
1,ARR (End of Period),$34.2M,arr,USD,Q2 2025,point_in_time,1,table
2,Gross Margin,78%,gross_margin,NaN,Q2 2025,quarterly,1,table
3,Net Dollar Retention,123%,net_revenue_retention,NaN,Q2 2025,ltm,1,table
4,Logo Churn (LTM),5.8%,logo_churn,NaN,Q2 2025,ltm,1,table
5,ARR per Full-Time Employee,$241k,NaN,USD,Q2 2025,point_in_time,1,table
6,Cash Balance,$19.6M,cash,USD,Q2 2025,point_in_time,1,table
7,Monthly Net Burn,($0.75M),net_burn,USD,Q2 2025,monthly,1,table
8,Total Headcount,142,headcount,NaN,Q2 2025,point_in_time,1,table
9,Recognized Revenue,$8.4M,revenue,USD,Q2 2025,quarterly,1,commentary


## Stage 4 — Normalize: deterministic rules (`src/normalize.py`)

Company resolved (original kept in `company_as_reported`), canonical claims validated
against the dictionary, period split into sortable quarter/year, duplicates handled.

In [5]:
from src.normalize import normalize

df = normalize([(SAMPLE, extraction)], settings["metric_definitions"])
df[["company", "company_as_reported", "canonical_metric", "verbatim_label", "value",
    "quarter", "year", "non_canonical", "superseded"]]

,company,company_as_reported,canonical_metric,verbatim_label,value,quarter,year,non_canonical,superseded
0,NovaCloud,NovaCloud Analytics Inc.,revenue,Recognized Revenue (USD),$8.4M,2,2025,False,False
1,NovaCloud,NovaCloud Analytics Inc.,arr,ARR (End of Period),$34.2M,2,2025,False,False
2,NovaCloud,NovaCloud Analytics Inc.,gross_margin,Gross Margin,78%,2,2025,False,False
3,NovaCloud,NovaCloud Analytics Inc.,net_revenue_retention,Net Dollar Retention,123%,2,2025,False,False
4,NovaCloud,NovaCloud Analytics Inc.,logo_churn,Logo Churn (LTM),5.8%,2,2025,False,False
5,NovaCloud,NovaCloud Analytics Inc.,NaN,ARR per Full-Time Employee,$241k,2,2025,True,False
6,NovaCloud,NovaCloud Analytics Inc.,cash,Cash Balance,$19.6M,2,2025,False,False
7,NovaCloud,NovaCloud Analytics Inc.,net_burn,Monthly Net Burn,($0.75M),2,2025,False,False
8,NovaCloud,NovaCloud Analytics Inc.,headcount,Total Headcount,142,2,2025,False,False
9,NovaCloud,NovaCloud Analytics Inc.,revenue,Recognized Revenue,$8.4M,2,2025,False,True


## Stage 5 — Checks: provenance guard + data-quality (`src/checks.py`)

In [6]:
from src import checks

flags = checks.run_all(df, {SAMPLE.name: pages})
pd.DataFrame(flags) if flags else print("no flags — every value verified against the source text")

16:37:42 INFO    checks: checks complete: {'info': 3}


,severity,source_file,company,metric,period,detail
0,info,NovaCloud_Q2_2025.pdf,NovaCloud,arr,Q2 2025,same value restated under multiple labels/loca...
1,info,NovaCloud_Q2_2025.pdf,NovaCloud,gross_margin,Q2 2025,same value restated under multiple labels/loca...
2,info,NovaCloud_Q2_2025.pdf,NovaCloud,revenue,Q2 2025,same value restated under multiple labels/loca...


That is the whole pipeline for one document. The batch run (`python -m src.pipeline --input data`)
does exactly this per PDF, then adds the cross-document work: precedence between overlapping
sources, cross-source consistency checks, and the three output files.